# Centre-only qualification model

Train on Centre historical applications only, inspect held-out performance and diagnostics, then score both datasets as `q`. `decision_octroi` is the historical label used for training, not an independent fairness ground truth.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
ELOIGNEES = ['Bas-Saint-Laurent', 'Cote-Nord', 'Gaspesie-Iles-de-la-Madeleine']

def groupe_region(df):
    return np.where(df['region_administrative'].isin(ELOIGNEES), 'Eloignee', 'Centre')

historique = pd.read_csv('data/donnees_demandes.csv')
evaluation = pd.read_csv('data/candidats_evaluation.csv')
historique['groupe'] = groupe_region(historique)
evaluation['groupe'] = groupe_region(evaluation)

features = ['cote_r_equivalent', 'revenu_familial_estime', 'heures_travail_semaine',
            'premiere_generation_universitaire', 'programme_etudes']
numeric_features = features[:-1]
categorical_features = ['programme_etudes']
centre = historique.loc[historique['groupe'].eq('Centre')].copy()
print(f'Historical rows: {len(historique):,}; evaluation rows: {len(evaluation):,}; centre training pool: {len(centre):,}')

Historical rows: 10,000; evaluation rows: 4,000; centre training pool: 6,000


## Centre holdout evaluation

In [2]:
train_idx, test_idx = train_test_split(
    centre.index, test_size=0.2, random_state=RANDOM_STATE,
    stratify=centre['decision_octroi'])
train = historique.loc[train_idx]
test = historique.loc[test_idx]
preprocess = ColumnTransformer([
    ('numeric', 'passthrough', numeric_features),
    ('program', OneHotEncoder(handle_unknown='ignore'), categorical_features),
])
model = make_pipeline(preprocess, HistGradientBoostingClassifier(random_state=RANDOM_STATE))
model.fit(train[features], train['decision_octroi'])
test_q = model.predict_proba(test[features])[:, 1]
print(f'Centre held-out accuracy: {accuracy_score(test["decision_octroi"], test_q >= 0.5):.4f}')
print(f'Centre held-out AUC: {roc_auc_score(test["decision_octroi"], test_q):.4f}')
calibration = pd.DataFrame({'q': test_q, 'observed': test['decision_octroi'].to_numpy()})
calibration['bin'] = pd.qcut(calibration['q'], q=10, labels=False, duplicates='drop') + 1
print('Calibration by predicted-probability decile:')
print(calibration.groupby('bin', observed=True).agg(n=('q','size'), predicted_grant_rate=('q','mean'), observed_grant_rate=('observed','mean')).to_string(float_format=lambda x: f'{x:.4f}'))

Centre held-out accuracy: 0.8633
Centre held-out AUC: 0.9426
Calibration by predicted-probability decile:
       n  predicted_grant_rate  observed_grant_rate
bin                                                
1    120                0.0001               0.0083
2    120                0.0017               0.0083
3    120                0.0174               0.0917
4    120                0.1138               0.1500
5    120                0.3362               0.3333
6    120                0.5750               0.6083
7    120                0.8190               0.7250
8    120                0.9656               0.9333
9    120                0.9970               0.9750
10   120                0.9998               1.0000


## Score all applicants and inspect diagnostics

In [3]:
historique['q'] = model.predict_proba(historique[features])[:, 1]
evaluation['q'] = model.predict_proba(evaluation[features])[:, 1]
print('Historical mean q and actual grant rate by group:')
print(historique.groupby('groupe').agg(mean_q=('q','mean'), actual_decision_octroi_rate=('decision_octroi','mean'), n=('q','size')).to_string(float_format=lambda x: f'{x:.4f}'))

r_edges = [15, 25, 27, 29, 31, 40]
historique['r_bin'] = pd.cut(historique['cote_r_equivalent'], bins=r_edges, right=False, include_lowest=True)
r_diagnostics = historique.groupby(['r_bin','groupe'], observed=False).agg(
    count=('decision_octroi','size'), mean_decision_octroi=('decision_octroi','mean'), mean_q=('q','mean'))
print('Historical R-score bin diagnostics by group:')
print(r_diagnostics.to_string(float_format=lambda x: f'{x:.4f}'))
print('Evaluation mean q by group:')
print(evaluation.groupby('groupe').agg(mean_q=('q','mean'), n=('q','size')).to_string(float_format=lambda x: f'{x:.4f}'))

permutation = permutation_importance(model, test[features], test['decision_octroi'],
                                     scoring='roc_auc', n_repeats=10, random_state=RANDOM_STATE)
importance = pd.DataFrame({'feature': features, 'importance_mean': permutation.importances_mean,
                           'importance_std': permutation.importances_std}).sort_values('importance_mean', ascending=False)
print('Permutation importance (ROC AUC decrease; held-out Centre rows):')
print(importance.to_string(index=False, float_format=lambda x: f'{x:.5f}'))

historique.drop(columns=['r_bin']).to_csv('data/q_historique.csv', index=False)
evaluation.to_csv('data/q_evaluation.csv', index=False)
print('Saved data/q_historique.csv and data/q_evaluation.csv')

Historical mean q and actual grant rate by group:
          mean_q  actual_decision_octroi_rate     n
groupe                                             
Centre    0.4835                       0.4837  6000
Eloignee  0.4443                       0.2730  4000
Historical R-score bin diagnostics by group:
                   count  mean_decision_octroi  mean_q
r_bin    groupe                                       
[15, 25) Centre      971                0.0041  0.0043
         Eloignee    841                0.0012  0.0048
[25, 27) Centre     1280                0.1102  0.1066
         Eloignee   1021                0.0264  0.1990
[27, 29) Centre     1558                0.4538  0.4537
         Eloignee    957                0.1745  0.4785
[29, 31) Centre     1221                0.8894  0.8915
         Eloignee    759                0.6522  0.9112
[31, 40) Centre      970                0.9938  0.9952
         Eloignee    422                0.9526  0.9964
Evaluation mean q by group:
         

Permutation importance (ROC AUC decrease; held-out Centre rows):
                          feature  importance_mean  importance_std
                cote_r_equivalent          0.42217         0.01097
           revenu_familial_estime          0.01516         0.00235
           heures_travail_semaine          0.01416         0.00215
premiere_generation_universitaire          0.00106         0.00036
                 programme_etudes          0.00026         0.00079
Saved data/q_historique.csv and data/q_evaluation.csv
